##### Import statements:

In [24]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot
import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.autograd import Variable
from torch.utils.data import DataLoader
import torch.multiprocessing as mp
from multiprocessing import set_start_method
import time
import datetime
import socket
from ws.test_module import Mdl, init_train_mdl_proc
from analysis_metadata.analysis_metadata import Metadata, write_metadata, increment_dir_name

##### Set context:

In [2]:
hostname = socket.gethostname()
set_start_method('spawn')

##### Define parameters:

In [11]:
# Data parameters:
n_feat = 80
n_obs = 160
n_classes = 2
data_noise = 0.1

# Model parameters:
n_inp = n_feat
n_hidden = 100
sigma_init = 1
sigma_noise = 0.1
batch_size = 64
n_epochs = 50
lr = 0.001
beta_ces = 10**np.arange(0, 3, 1)
beta_sp = 1
p_norm = 2

# Compute paramters:
gpu = True
n_cores = 5

# Output params:
save_output = True
if 'rc.zi.columbia.edu' in hostname:
    base_output_directory = os.path.join('/', 'mnt', 'smb', 'locker', 'issa-locker', 'users', 'Dan', 'code', 'ws', 'results', 'speed_tests')

##### Generate data:

In [12]:
if __name__ == '__main__':
    # Define centroid for each class:
    mus = []
    for i in np.arange(n_classes):
        curr_mu = np.random.randn(n_feat).astype(np.float32)
        mus.append(curr_mu)
    
    # Define covariance matrix:
    C = np.random.randn(n_feat, n_feat).astype(np.float32)
    
    # Generate labels:
    labels = np.random.choice(np.arange(n_classes), n_obs)
    labels = np.expand_dims(labels, axis=1)
    
    # Generate data:
    X = np.array([mus[x] for x in np.squeeze(labels)])
    
    # Add noise:
    Eps = np.random.randn(n_obs, n_feat).astype(np.float32)
    Eps = np.matmul(C, Eps.T).T 
    X = X + Eps
    
    # Convert data to torch:
    X_torch = Variable(torch.from_numpy(X))
    labels_torch = Variable(torch.from_numpy(labels)).long()
    
    # Move to gpu if necessary:
    if gpu:
        X_torch = X_torch.to('cuda')
        labels_torch = labels_torch.to('cuda')

##### Iterate over training episodes:

In [13]:
if __name__ == '__main__':
    start_train = time.time()

    manager = mp.Manager()
    results_dict = manager.dict()
    processes = []
    for b, beta_ce in enumerate(beta_ces):
    
        #print('Running hyperparameter value {} out of {}...'.format(b+1, len(beta_ces)))
        p = mp.Process(target=init_train_mdl_proc, args=(X_torch, labels_torch, n_hidden, results_dict, b), 
                       kwargs={'n_epochs':n_epochs, 'beta_ce':beta_ce, 'beta_sp':beta_sp, 'sigma_init':sigma_init, 'sigma_noise':sigma_noise, 'batch_size':batch_size, 'gpu':gpu})
        p.start()
        processes.append(p)

    for p in processes:
        p.join()
    
    stop_train = time.time()

##### Compute mean function call duration:

In [14]:
if __name__ == '__main__':
    mean_fcn_call_dur = np.mean([dict(results_dict)[x] for x in dict(results_dict)])
    total_dur = stop_train - start_train 
    print('Mean fcn call duration = {}'.format(mean_fcn_call_dur))
    print('Training duration : {} s'.format(total_dur))

Mean fcn call duration = 16.11649759610494
Training duration : 22.02548885345459 s


##### Save output:

In [26]:
if __name__ == '__main__' and save_output:

    # Create output directory if necessary:
    if not os.path.exists(base_output_directory):
        pathlib.Path(base_output_directory).mkdir(parents=True, exist_ok=True)
    curr_output_directory=increment_dir_name(base_output_directory, 'run')
    if not os.path.exists(curr_output_directory):
        pathlib.Path(curr_output_directory).mkdir(parents=True, exist_ok=True)

    # Save results to metadata file:
    M = Metadata()
    script_path = os.path.join(os.getcwd(), 'torch_speed_test_iter_par_proc.ipynb')
    results_path = os.path.join(curr_output_directory, 'torch_speed_test_iter_par_proc_results.json')
    M.add_input(script_path)
    M.add_param('hostname', hostname)
    M.add_param('beta_ces', list(beta_ces))
    M.add_param('n_feat', n_feat)
    M.add_param('n_obs', n_obs)
    M.add_param('gpu', gpu)
    M.add_param('n_cores', n_cores)
    M.add_param('mean_fcn_call_dur', mean_fcn_call_dur)
    M.add_output(results_path)
    now = datetime.datetime.now()
    M.date = now.strftime('%Y-%m-%d')
    M.time = now.strftime('%H:%M:%S')
    M.duration = total_dur
    write_metadata(M, results_path, debug=True)